# CRLB-Based InSAR Workflow Tutorial (English)

This notebook demonstrates how to run the **CRLB-based coherence path** with the current `insar_pipeline` codebase and visualize each key generated artifact.

Workflow:
1. Crop (`lat/lon` + interferogram files)
2. Build dataset with `coherence_source="computed_crlb"`
3. Train/predict
4. Compute score
5. Generate geocoded outputs

At each stage, we plot generated files (`.npy` via `matplotlib`, `.rdr/.cor` via GDAL).

In [ ]:
# --- Configuration (edit these paths before running) ---
from pathlib import Path

BASE_DIR = Path('/data6/WORKDIR/AmatriceSenDT22/merged/interferograms')
GEOM_REFERENCE_DIR = Path('/data6/WORKDIR/AmatriceSenDT22/merged/geom_reference')
OUTPUT_DIR = BASE_DIR / 'cropped'
STACK_ROOT = BASE_DIR  # set this to your stack root if different
NEXT_DATE = '20160821_20160902'

LAT_MIN, LAT_MAX = 42.625, 42.635
LON_MIN, LON_MAX = 13.28, 13.30

print('BASE_DIR:', BASE_DIR)
print('GEOM_REFERENCE_DIR:', GEOM_REFERENCE_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)


In [ ]:
# --- Imports ---
import pickle
import numpy as np
import matplotlib.pyplot as plt
from osgeo import gdal

from insar_pipeline.preprocess import CropConfig, batch_crop_filt_fine_cor
from insar_pipeline.dataset_builder import DatasetConfig, build_and_save_dataset
from insar_pipeline.modeling import TrainingConfig, run_training_and_prediction
from insar_pipeline.scoring import ScoreConfig, compute_and_save_score
from insar_pipeline.output_products import OutputConfig, generate_geocoded_outputs


In [ ]:
# --- Visualization helpers ---
def show_npy(path, title=None, cmap='viridis'):
    arr = np.load(path)
    if arr.ndim == 3:
        arr2d = arr[:, :, 0]
    else:
        arr2d = arr

    plt.figure(figsize=(7, 3.5))
    im = plt.imshow(arr2d, cmap=cmap)
    plt.colorbar(im, shrink=0.8)
    plt.title(title or f'{path.name} | shape={arr.shape} | dtype={arr.dtype}')
    plt.tight_layout()
    plt.show()


def show_raster(path, band=1, title=None, cmap='viridis'):
    ds = gdal.Open(str(path), gdal.GA_ReadOnly)
    if ds is None:
        raise FileNotFoundError(f'Cannot open raster: {path}')
    arr = ds.GetRasterBand(band).ReadAsArray()

    plt.figure(figsize=(7, 3.5))
    im = plt.imshow(arr, cmap=cmap)
    plt.colorbar(im, shrink=0.8)
    plt.title(title or f'{path.name} (band {band}) | shape={arr.shape} | dtype={arr.dtype}')
    plt.tight_layout()
    plt.show()


## 1) Crop input files and inspect cropped `lat/lon`

In [ ]:
crop_outputs = batch_crop_filt_fine_cor(
    CropConfig(
        base_path=BASE_DIR,
        geom_reference_path=GEOM_REFERENCE_DIR,
        output_base_path=OUTPUT_DIR,
        lat_min=LAT_MIN,
        lat_max=LAT_MAX,
        lon_min=LON_MIN,
        lon_max=LON_MAX,
    )
)
print('Number of cropped interferogram files:', len(crop_outputs))
print('Example output:', crop_outputs[0] if crop_outputs else 'None')

lat_file = OUTPUT_DIR / 'lat_cropped.rdr'
lon_file = OUTPUT_DIR / 'lon_cropped.rdr'
print('lat_file:', lat_file)
print('lon_file:', lon_file)


In [ ]:
show_raster(lat_file, band=1, title='lat_cropped.rdr (band 1)', cmap='coolwarm')
show_raster(lon_file, band=1, title='lon_cropped.rdr (band 1)', cmap='coolwarm')


## 2) Build dataset with CRLB coherence and visualize saved `.npy` files

In [ ]:
dataset_dir = build_and_save_dataset(
    DatasetConfig(
        cropped_dir=OUTPUT_DIR,
        output_dir=OUTPUT_DIR,
        input_source='stack_int',
        stack_root=STACK_ROOT,
        coherence_source='computed_crlb',
        persist_computed_cor=True,
    )
)
print('dataset_dir:', dataset_dir)


In [ ]:
for name in ['data.npy', 'data_std.npy', 'geninue.npy', 'geninue_std.npy']:
    p = dataset_dir / name
    print(name, 'exists =', p.exists())
    if p.exists():
        arr = np.load(p, mmap_mode='r')
        print('   shape:', arr.shape, 'dtype:', arr.dtype)

with open(dataset_dir / 'dates.pkl', 'rb') as f:
    dates = pickle.load(f)
print('dates length:', len(dates), 'sample:', dates[:3])


In [ ]:
show_npy(dataset_dir / 'data.npy', title='data.npy (first time slice)')
show_npy(dataset_dir / 'data_std.npy', title='data_std.npy (first time slice)')
show_npy(dataset_dir / 'geninue.npy', title='geninue.npy')
show_npy(dataset_dir / 'geninue_std.npy', title='geninue_std.npy')


## 3) Train/predict and visualize `future_predictions.npy`

In [ ]:
predict_dir = run_training_and_prediction(
    TrainingConfig(
        dataset_dir=dataset_dir,
        output_dir=OUTPUT_DIR,
        next_date=NEXT_DATE,
    )
)
print('predict_dir:', predict_dir)


In [ ]:
fp = predict_dir / 'future_predictions.npy'
print(fp, 'exists =', fp.exists())
if fp.exists():
    arr = np.load(fp, mmap_mode='r')
    print('shape:', arr.shape, 'dtype:', arr.dtype)
show_npy(fp, title='future_predictions.npy', cmap='magma')


## 4) Compute score and visualize `score.npy`

In [ ]:
score_path = compute_and_save_score(
    ScoreConfig(
        dataset_dir=dataset_dir,
        predict_dir=predict_dir,
        score_filename='score.npy',
    )
)
print('score_path:', score_path)


In [ ]:
show_npy(score_path, title='score.npy', cmap='RdBu_r')


## 5) Generate geocoded outputs and visualize `.cor` products

In [ ]:
output_files = generate_geocoded_outputs(
    OutputConfig(
        predict_dir=predict_dir,
        lat_file=lat_file,
        lon_file=lon_file,
        subset_params='-l 42.625 42.635 -L 13.28 13.30',
    )
)
print('Generated output files:')
for p in output_files:
    print(' -', p)


In [ ]:
for p in sorted(predict_dir.glob('geo_*.cor')):
    print('Showing', p.name)
    show_raster(p, band=1, title=f'{p.name} (band 1)', cmap='viridis')

for p in sorted(predict_dir.glob('*final.cor')):
    print('Showing', p.name)
    show_raster(p, band=1, title=f'{p.name} (band 1)', cmap='viridis')


## Notes

- This notebook focuses on the **CRLB** path (`coherence_source='computed_crlb'`).
- If your stack root differs from `BASE_DIR`, update `STACK_ROOT` accordingly.
- If geocoding tools (`geocode.py`, `subset.py`, `save_gdal.py`) are unavailable, run the output stage inside your MintPy/ISCE environment.